### 2022 - 2025

In [1]:
import pandas as pd
import numpy as np

# 1. Loading data
df_vitals = pd.read_csv("Datasets/df_param_vit_2022_2025.csv", sep=',', low_memory=False)
df_ioa = pd.read_csv("Datasets/df_ioa_2022_2025.csv", sep=',', dtype={'nda': str})

# 2. cleaning nda
def clean_nda(series):
    # trnansforming in text, removing .0 and stripping spaces
    return series.astype(str).str.replace(r'\.0$', '', regex=True).str.strip()

print("🧹 nda cleaning ...")
df_vitals['nda'] = clean_nda(df_vitals['nda'])
df_ioa['nda'] = clean_nda(df_ioa['nda'])

# 3. Junction analysis (does it work?)
nda_vitals = set(df_vitals['nda'].unique())
nda_ioa = set(df_ioa['nda'].unique())
commons = nda_vitals.intersection(nda_ioa)

print(f"\n✅ NDA in common : {len(commons):,}".replace(',', ' '))

if len(commons) == 0:
    print("\n🧐 Always 0 ? lets take a look at them :")
    # On affiche les 3 premiers pour comparer visuellement
    print(f"Examples VITALS : {list(nda_vitals)[:3]}")
    print(f"Examples IOA    : {list(nda_ioa)[:3]}")

# 4. Checking each df for date and hospital columns
def audit_exhaustif(df, nom_df, col_date, col_hospital):
    print(f"\n" + "="*70)
    print(f"📊 Annual analysis : {nom_df}")
    print(f"="*70)

    # Conversion date
    # we use mapped column to avoid creating heavy columns in the original df, we will drop them after
    df['temp_date'] = pd.to_datetime(df[col_date], errors='coerce')
    df['annee'] = df['temp_date'].dt.year

    # check hospital column
    if col_hospital not in df.columns:
        print(f"⚠️ Error : Column '{col_hospital}' not in this file.")
        return

    # Loop on each hospital (SA, PEL)
    for h in df[col_hospital].unique():
        if pd.isna(h): continue # ingnoring nan in hospital name

        print(f"\n🏥 Hospital : {h}")
        print("-" * 30)

        # filtering and counting per year
        stats_annee = df[df[col_hospital] == h]['annee'].value_counts().sort_index()

        if not stats_annee.empty:
            # clean display: year | Nulber of lines
            for annee, count in stats_annee.items():
                print(f"Year {int(annee)} : {count:,} lines".replace(',', ' '))

            print(f"\nTOTAL {h} : {stats_annee.sum():,} lines".replace(',', ' '))
        else:
            print(f"⚠️ No year data valid for this hospital {h}.")



🧹 nda cleaning ...

✅ NDA in common : 111 084


### 2022 only

In [2]:
# import pandas as pd
# import numpy as np
#
# # 1. Loading data
# df_vitals = pd.read_csv("df_param_vit_subset22pel.csv", sep=',', low_memory=False)
# df_ioa = pd.read_csv("df_ioa_subset22pel.csv", sep=',', dtype={'nda': str})
#
# # 2. cleaning nda
# def clean_nda(series):
#     # trnansforming in text, removing .0 and stripping spaces
#     return series.astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
#
# print("🧹 nda cleaning ...")
# df_vitals['nda'] = clean_nda(df_vitals['nda'])
# df_ioa['nda'] = clean_nda(df_ioa['nda'])
#
# # 3. Junction analysis (does it work?)
# nda_vitals = set(df_vitals['nda'].unique())
# nda_ioa = set(df_ioa['nda'].unique())
# commons = nda_vitals.intersection(nda_ioa)
#
# print(f"\n✅ NDA in common : {len(commons):,}".replace(',', ' '))
#
# if len(commons) == 0:
#     print("\n🧐 Always 0 ? lets take a look at them :")
#     # On affiche les 3 premiers pour comparer visuellement
#     print(f"Examples VITALS : {list(nda_vitals)[:3]}")
#     print(f"Examples IOA    : {list(nda_ioa)[:3]}")
#
# # 4. Checking each df for date and hospital columns
# def audit_exhaustif(df, nom_df, col_date, col_hospital):
#     print(f"\n" + "="*70)
#     print(f"📊 Annual analysis : {nom_df}")
#     print(f"="*70)
#
#     # Conversion date
#     # we use mapped column to avoid creating heavy columns in the original df, we will drop them after
#     df['temp_date'] = pd.to_datetime(df[col_date], errors='coerce')
#     df['annee'] = df['temp_date'].dt.year
#
#     # check hospital column
#     if col_hospital not in df.columns:
#         print(f"⚠️ Error : Column '{col_hospital}' not in this file.")
#         return
#
#     # Loop on each hospital (SA, PEL)
#     for h in df[col_hospital].unique():
#         if pd.isna(h): continue # ingnoring nan in hospital name
#
#         print(f"\n🏥 Hospital : {h}")
#         print("-" * 30)
#
#         # filtering and counting per year
#         stats_annee = df[df[col_hospital] == h]['annee'].value_counts().sort_index()
#
#         if not stats_annee.empty:
#             # clean display: year | Nulber of lines
#             for annee, count in stats_annee.items():
#                 print(f"Year {int(annee)} : {count:,} lines".replace(',', ' '))
#
#             print(f"\nTOTAL {h} : {stats_annee.sum():,} lines".replace(',', ' '))
#         else:
#             print(f"⚠️ No year data valid for this hospital {h}.")
#


In [3]:
df_vitals["date_hour_adm_vitals_file"] = df_vitals["date_adm_vitals"]


def generer_tableau_comparatif_direct(df, nom_df, col_date, col_hopital):
    print(f"\n{'='*25} {nom_df} {'='*25}")

    # OPTIMISATION : we create a temporary column for year to avoid doing it multiple times in the crosstab, we will drop it after
    annees = pd.to_datetime(df[col_date], errors='coerce').dt.year

    # Cross table with margins to have totals, we will drop the total line after if we want to keep only years
    tableau = pd.crosstab(annees, df[col_hopital], margins=True, margins_name="TOTAL")

    # displaying the table
    print(tableau)
    return tableau

# Lancement
tab_vitals = generer_tableau_comparatif_direct(df_vitals, "TABLE : VITALS", "date_hour_adm_vitals_file", "hospital")
tab_ioa = generer_tableau_comparatif_direct(df_ioa, "TABLE : IOA", "date_adm_ioa_file", "hospital")


========================= TABLE : VITALS =========================


hospital                      PEL   TOTAL
date_hour_adm_vitals_file                
2022                        30581   30581
2023                        29146   29146
2024                        28981   28981
2025                        24770   24770
TOTAL                      113478  113478

========================= TABLE : IOA =========================
hospital              PEL   TOTAL
date_adm_ioa_file                
2022                43431   43431
2023                37965   37965
2024                41774   41774
2025                45798   45798
TOTAL              168968  168968


In [4]:
# ===================================================
# Admission Date Comparison between IOA file and Vitals
# ===================================================
# These should match; only triage date and measurement datetime may differ

# --- ADMISSION DATE CONSISTENCY CHECK ---

# 1. Prepare small dataframes with only necessary columns
df_ioa_dates = df_ioa[['nda', 'date_adm_ioa_file']].copy()
df_vitals_dates = df_vitals[['nda', 'date_hour_adm_vitals_file']].copy()

# 2. Quick cleaning of patient IDs for merge
df_ioa_dates['nda'] = df_ioa_dates['nda'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
df_vitals_dates['nda'] = df_vitals_dates['nda'].astype(str).str.replace(r'\.0$', '', regex=True).str.strip()

# 3. Convert to datetime
df_ioa_dates['date_adm_ioa_file'] = pd.to_datetime(df_ioa_dates['date_adm_ioa_file'], errors='coerce')
df_vitals_dates['date_hour_adm_vitals_file'] = pd.to_datetime(df_vitals_dates['date_hour_adm_vitals_file'],
                                                                   errors='coerce')

# 4. Temporary merge on patient ID to compare common rows
df_compare = pd.merge(df_ioa_dates, df_vitals_dates, on='nda', how='inner')

# 5. Compute difference in minutes
df_compare['diff_minutes'] = (df_compare['date_adm_ioa_file'] - df_compare['date_hour_adm_vitals_file']).dt.total_seconds() / 60

print("=== ADMISSION DATE OFFSET ANALYSIS ===")
print(f"Number of records compared: {len(df_compare)}")
print(f"Mean difference: {df_compare['diff_minutes'].mean():.2f} minutes")
print(f"Median difference: {df_compare['diff_minutes'].median():.2f} minutes")
print(f"Max difference: {df_compare['diff_minutes'].max():.2f} minutes")

# 6. Count “perfect matches” (< 1 min difference)
perfect_matches = len(df_compare[df_compare['diff_minutes'].abs() <= 1])
print(f"Records with < 1 min difference: {perfect_matches} ({perfect_matches / len(df_compare) * 100:.1f}%)")

if perfect_matches / len(df_compare) < 0.8:
    print("\n⚠️ WARNING: Too many discrepancies. Software may not be synchronized.")
else:
    print("\n✅ GOOD: Admission dates are consistent between the two files.")

=== ADMISSION DATE OFFSET ANALYSIS ===
Number of records compared: 111084
Mean difference: 0.00 minutes
Median difference: 0.00 minutes
Max difference: 0.00 minutes
Records with < 1 min difference: 111084 (100.0%)

✅ GOOD: Admission dates are consistent between the two files.


In [5]:
# 1. EXPLO DURATION TRIAGE

print(df_ioa['duration_triage_ioa_min'].dtype)


# Statistiques demandées (Min, Max, Moyenne, Médiane, Quartiles)
stats = df_ioa['duration_triage_ioa_min'].describe(percentiles=[.25, .5, .75])

print("Full stat of triage duration :")
print(f"- Minimum : {stats['min']:.2f} min")
print(f"- Maximum : {stats['max']:.2f} min")
print(f"- Mean : {stats['mean']:.2f} min")
print(f"- Médian : {stats['50%']:.2f} min")
print(f"- Q1 (25%) : {stats['25%']:.2f} min")
print(f"- Q3 (75%) : {stats['75%']:.2f} min")

float64
Full stat of triage duration :
- Minimum : -466559.00 min
- Maximum : 466591.00 min
- Mean : -2809.86 min
- Médian : 5.00 min
- Q1 (25%) : -167032.00 min
- Q3 (75%) : 128167.00 min


In [6]:
# --- ANOMALY DETECTION → SET TO NA ---

# 1. Identify anomalies in triage duration (<2 min or >30 min)
mask_anomalie = (df_ioa['duration_triage_ioa_min'] < 2) | (df_ioa['duration_triage_ioa_min'] > 30)

# 2. Create a new dataframe with only the anomalous rows for inspection
df_anomalies = df_ioa.loc[mask_anomalie, [
    'date_adm_ioa_file',
    'date_hour_triage_begin',
    'date_hour_triage_end',
    'duration_triage_ioa_min'
]]

# 3. Display anomalies
nb_anomalies = len(df_anomalies)
print(f"--- ANOMALY ANALYSIS ---")
print(f"NUMBER OF ROWS WITH DURATION < 2 min or > 30 min: {nb_anomalies}")
print(f"\nPreview of anomalous rows:")
print(df_anomalies.sort_values(by='duration_triage_ioa_min').head(300))

# 4. Analyze long triage durations (>30 min)
df_long_triage = df_ioa[df_ioa['duration_triage_ioa_min'] > 30].copy()

repartition_tri_long = df_long_triage['triage'].value_counts(dropna=False).sort_index()
print(f"\n--- TRIAGE SCORE ANALYSIS FOR DURATION > 30 MIN ---")
print(f"Total number of patients affected: {len(df_long_triage)}")
print("\nScore distribution:")
print(repartition_tri_long)

print("\nTop 5 chief complaints for long triage durations:")
print(df_long_triage['chief_complaint'].value_counts().head(5))

# 5. Set anomalous values to NaN instead of removing rows
df_ioa.loc[mask_anomalie, 'duration_triage_ioa_min'] = np.nan
print(f"\n✅ {nb_anomalies} anomalous durations set to NaN.")
print(f"Remaining valid durations: {df_ioa['duration_triage_ioa_min'].notna().sum():,}")

--- ANOMALY ANALYSIS ---
NUMBER OF ROWS WITH DURATION < 2 min or > 30 min: 61482

Preview of anomalous rows:
          date_adm_ioa_file  ... duration_triage_ioa_min
82676   2024-01-12 16:47:00  ...               -466559.0
82698   2024-01-12 19:34:00  ...               -466559.0
82645   2024-01-12 11:27:00  ...               -466558.0
82673   2024-01-12 16:29:00  ...               -466558.0
82628   2024-01-12 03:09:00  ...               -466558.0
...                     ...  ...                     ...
44567   2023-01-12 09:52:00  ...               -465114.0
44563   2023-01-12 09:27:00  ...               -465114.0
124574  2025-01-12 21:59:00  ...               -465114.0
44566   2023-01-12 09:46:00  ...               -465114.0
44561   2023-01-12 09:00:00  ...               -465114.0

[300 rows x 4 columns]

--- TRIAGE SCORE ANALYSIS FOR DURATION > 30 MIN ---
Total number of patients affected: 30259

Score distribution:
triage
1.0      217
2.0     6073
3.0    11893
4.0     9298
5.0     2

In [7]:
#===========================================
##### MERGING VITAL SIGNS AND IOA FILE #####
#===========================================

In [8]:
import pandas as pd

# ===============================
# 1. Data preparation & cleaning
# ===============================
print("--- 1. Data preparation & technical cleaning ---")

# Ensure NDA and hospital columns are strings, trimmed, and standardized
def clean_nda(series):
    return series.astype(str).str.replace(r'\.0$', '', regex=True).str.strip()

def clean_hospital(series):
    return series.astype(str).str.strip().str.upper()

df_ioa['nda'] = clean_nda(df_ioa['nda'])
df_vitals['nda'] = clean_nda(df_vitals['nda'])

df_ioa['hospital'] = clean_hospital(df_ioa['hospital'])
df_vitals['hospital'] = clean_hospital(df_vitals['hospital'])

print(f"Unique NDA in IOA : {df_ioa['nda'].nunique()}")
print(f"Unique NDA in Vitals : {df_vitals['nda'].nunique()}")

# ===============================
# 2. Merge IOA and Vitals
# ===============================
df = pd.merge(
    df_ioa,
    df_vitals,
    on=['nda', 'hospital'],
    how='left',   # keep all rows to see missing matches
    indicator=True
)

# Optional: replace _merge labels for clarity
df['_merge'] = df['_merge'].replace({
    'left_only': 'ioa_only',
    'right_only': 'vitals_only',
    'both': 'ioa_and_vitals'
})

print("\n--- Merge diagnostic ---")
print(df['_merge'].value_counts())

# ===============================
# 3. Unify admission dates
# ===============================
# If the dates match perfectly, we can pick one column
df['date_adm_final'] = df['date_adm_ioa_file'].fillna(df['date_hour_adm_vitals_file'])
df['date_adm_final'] = pd.to_datetime(df['date_adm_final'], errors='coerce')

print("\n--- Dataset ready for analysis ---")
print(f"Final columns: {list(df.columns)}")
print(f"Number of rows: {len(df)}")

--- 1. Data preparation & technical cleaning ---


Unique NDA in IOA : 168968
Unique NDA in Vitals : 113478



--- Merge diagnostic ---
_merge
ioa_and_vitals    111084
ioa_only           57884
vitals_only            0
Name: count, dtype: int64

--- Dataset ready for analysis ---
Final columns: ['nda', 'age_ioa', 'sex_ioa', 'hospital', 'year', 'date_adm_ioa_file', 'date_hour_triage_begin', 'date_hour_triage_end', 'duration_triage_ioa_min', 'triage', 'triage_raw', 'transport', 'chief_complaint', 'anam_ioa', 'atcd_ioa', 'ttt_adm_ioa_file', 'admission_summary_ioa', 'evolution_ioa', 'date_adm_vitals', 'urine_dipstick', 'sbp', 'dbp', 'hr', 'temp', 'sat', 'rr', 'o2_flow', 'cap_blood_sugar', 'hemocue', 'gcs', 'pain', 'breathalyzer', 'pupil_right', 'pupil_left', 'source_file', 'date_hour_adm_vitals_file', '_merge', 'date_adm_final']
Number of rows: 168968


/tmp/ipykernel_604948/168882949.py:36: FutureWarning:

The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.



In [9]:
# --- CLEANING AND REORGANIZATION (Optimized Combo) ---

# 1. Define columns to keep and drop
columns_order = [
    'nda', 'date_adm_final', 'hospital', 'transport',
    'date_hour_triage_begin', 'date_hour_triage_end', 'duration_triage_ioa_min', 'chief_complaint', 'triage', 'triage_raw', 'atcd_ioa', 'anam_ioa'
]

columns_to_drop = [
    'age_ioa', 'sex_ioa', 'source_file', 'source_file_x', 'source_file_y',
    'temp_date_x', 'temp_date_y', 'annee_x', 'annee_y',
    'date_adm_ioa_file', 'date_hour_adm_vitals_file', '_merge'
]

# 2. Immediately clean RAM (in-place)
# Drop unnecessary columns before doing anything else
present_cols = [c for c in columns_to_drop if c in df.columns]
df.drop(columns=present_cols, inplace=True)

# 3. Dynamically identify remaining vital signs
# (Those not in the header order and not already dropped)
vital_signs = [c for c in df.columns if c not in columns_order]

# 4. Reorganize WITHOUT .copy()
# Create final order while checking columns exist
final_order = [c for c in columns_order if c in df.columns] + vital_signs
df = df[final_order]  # No .copy(), just reindex

print(f"✅ Reorganization done for {len(df)} patients.")
print(f"📊 Final columns ({len(df.columns)}) : {df.columns.tolist()}")

# 5. SAFE DISPLAY (to avoid huge file preview)
# Exclude heavy text columns (anam_ioa, atcd_ioa) from display
light_columns = [c for c in df.columns if 'anam' not in c and 'atcd' not in c]

print("\n--- Preview (structured data only) ---")
display(df[light_columns].head(10))

✅ Reorganization done for 168968 patients.
📊 Final columns (32) : ['nda', 'date_adm_final', 'hospital', 'transport', 'date_hour_triage_begin', 'date_hour_triage_end', 'duration_triage_ioa_min', 'chief_complaint', 'triage', 'triage_raw', 'atcd_ioa', 'anam_ioa', 'year', 'ttt_adm_ioa_file', 'admission_summary_ioa', 'evolution_ioa', 'date_adm_vitals', 'urine_dipstick', 'sbp', 'dbp', 'hr', 'temp', 'sat', 'rr', 'o2_flow', 'cap_blood_sugar', 'hemocue', 'gcs', 'pain', 'breathalyzer', 'pupil_right', 'pupil_left']

--- Preview (structured data only) ---


,nda,date_adm_final,hospital,transport,date_hour_triage_begin,date_hour_triage_end,duration_triage_ioa_min,chief_complaint,triage,triage_raw,year,ttt_adm_ioa_file,admission_summary_ioa,evolution_ioa,date_adm_vitals,urine_dipstick,sbp,dbp,hr,temp,sat,rr,o2_flow,cap_blood_sugar,hemocue,gcs,pain,breathalyzer,pupil_right,pupil_left
0,22030011617,2022-01-01 00:08:00,PEL,Pompiers,2022-01-01 00:19:00,2022-01-01 00:29:00,10.0,Victime d'agression physique,3.0,Urgent (médecin <1h),2022,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,22030011619,2022-01-01 00:17:00,PEL,SOS médecin,2022-01-01 00:28:00,2022-01-01 00:33:00,5.0,Détresse respiratoire aiguë majeure (FR >40/mi...,2.0,Très urgent (médecin <20min),2022,NaN,NaN,"01/01/2022 06:15 - MAUXION Anna, Infirmièr(e)...",2022-01-01 00:17:00,NaN,94.0,78.0,108.0,36.8,94.0,22.0,2.0,NaN,NaN,11.0,0.0,NaN,NaN,NaN
2,22030011622,2022-01-01 00:22:00,PEL,Spontanée,2022-01-01 00:34:00,2022-01-01 00:40:00,6.0,Mouvements involontaires anormaux,4.0,Peu urgent (médecin <2h),2022,NaN,NaN,NaN,2022-01-01 00:22:00,NaN,117.0,64.0,98.0,36.8,100.0,16.0,0.0,1.03,NaN,15.0,0.0,NaN,NaN,NaN
3,22030011625,2022-01-01 00:31:00,PEL,Spontanée,2022-01-01 00:49:00,2022-01-01 00:52:00,3.0,Demande de certificat médical,4.0,Peu urgent (médecin <2h),2022,NaN,NaN,NaN,2022-01-01 00:31:00,NaN,137.0,99.0,86.0,36.4,98.0,NaN,0.0,NaN,NaN,15.0,0.0,NaN,NaN,NaN
4,22030011629,2022-01-01 00:35:00,PEL,Spontanée,2022-01-01 00:42:00,2022-01-01 00:47:00,5.0,Anxiété sans troubles du comportement,4.0,Peu urgent (médecin <2h),2022,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,22030011634,2022-01-01 01:00:00,PEL,Spontanée,2022-01-01 01:14:00,2022-01-01 01:19:00,5.0,Dyspnée sans détresse respiratoire aiguë (FR <...,3.0,Urgent (médecin <1h),2022,NaN,NaN,NaN,2022-01-01 01:00:00,NaN,107.0,69.0,96.0,36.6,98.0,NaN,0.0,NaN,NaN,15.0,0.0,NaN,NaN,NaN
6,22030011639,2022-01-01 01:19:00,PEL,Spontanée,2022-01-01 01:38:00,2022-01-01 01:39:00,NaN,Douleur oculaire,4.0,Peu urgent (médecin <2h),2022,NaN,NaN,NaN,2022-01-01 01:19:00,NaN,109.0,65.0,96.0,37.1,97.0,NaN,0.0,NaN,NaN,15.0,8.0,NaN,NaN,NaN
7,22030011640,2022-01-01 01:31:00,PEL,Pompiers,2022-01-01 01:35:00,2022-01-01 01:37:00,2.0,Traumatisme crânien sans défaillance vitale,2.0,Très urgent (médecin <20min),2022,1G DOLITAPS + 100 MG OROZAMUDOL,NaN,NaN,2022-01-01 01:31:00,NaN,109.0,65.0,96.0,37.1,97.0,NaN,0.0,NaN,NaN,15.0,8.0,NaN,NaN,NaN
8,22030011642,2022-01-01 01:38:00,PEL,Spontanée,2022-01-01 01:42:00,2022-01-01 01:50:00,8.0,Plaie traumatique isolée,3.0,Urgent (médecin <1h),2022,1g dolitaps + 50 mg orozamudol,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,22030011643,2022-01-01 01:45:00,PEL,Spontanée,2022-01-01 02:01:00,2022-01-01 02:03:00,2.0,Traumatisme oeil ou orbite sans défaillance vi...,4.0,Peu urgent (médecin <2h),2022,NaN,NaN,NaN,2022-01-01 01:45:00,NaN,100.0,65.0,75.0,36.8,98.0,18.0,NaN,NaN,NaN,15.0,NaN,NaN,NaN,NaN


In [10]:
# CHECK THAT MAPPING WORKED WELL


def stats_rapides(df, colonne, nom_label):
    print(f"\n" + "="*40)
    print(f"📊 ANALYSE : {nom_label}")
    print("="*40)

    # 1. COMPUTATION OF COUNTS AND PERCENTAGES
    counts = df[colonne].value_counts(dropna=False)
    percents = (df[colonne].value_counts(dropna=False, normalize=True) * 100).round(1)

    # 2. SUMMARY TABLE CREATION
    df_stats = pd.DataFrame({
        'Patients': counts,
        'Pourcentage': percents.astype(str) + '%'
    }).sort_index()

    # 3. FORCE LIGHT DISPLAY
    print(df_stats.to_string())

    print("-" * 40)
    print(f"TOTAL : {len(df):,} dossiers".replace(',', ' '))

# --- EXÉCUTION ---
stats_rapides(df_ioa, 'triage', "NIVEAU DE TRI (Final)")
stats_rapides(df_ioa, 'triage_raw', "NIVEAU DE TRI (Brut)")


📊 ANALYSE : NIVEAU DE TRI (Final)
        Patients Pourcentage
triage                      
1.0         1196        0.7%
2.0        34843       20.6%
3.0        65047       38.5%
4.0        51980       30.8%
5.0        15876        9.4%
NaN           26        0.0%
----------------------------------------
TOTAL : 168 968 dossiers

📊 ANALYSE : NIVEAU DE TRI (Brut)
                                     Patients Pourcentage
triage_raw                                               
Infirmière <10min Médecin < 20min        9219        5.5%
Médecin <120min, puis IDE si besoin     16024        9.5%
Médecin <240min                          6452        3.8%
Médecin <60min, puis IDE si besoin       8319        4.9%
Médecin <90min, ouis IDE si besoin      17868       10.6%
Non urgent (médecin <3h)                 9424        5.6%
Peu urgent (médecin <2h)                35956       21.3%
Réanimation (médecin <1min)               497        0.3%
Sans délai (IDE et Médecin)               699        

In [11]:
#----------------------------------
# VARIABLES TYPE & NaN HARMONIZATION
#----------------------------------

import pandas as pd
import numpy as np

# 1️. Date Conversion
# Convert all date columns to datetime format (invalid values become NaT)
cols_dates = ['date_hour_vitals', 'date_tri_ioa_begin', 'date_tri_ioa_end', 'date_adm_final']

for col in cols_dates:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# 2️. Identifiers and Text Variables (String Handling)
# Ensure that all textual variables are properly treated as strings
# and harmonize missing values as np.nan
cols_strings = [
    'nda', 'chief_complaint', 'urine_dipstick', 'transport',
    'anam_ioa', 'atcd_ioa', 'hopital'
]

for col in cols_strings:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .replace(['nan', 'None', '', '<NA>', 'NaN'], np.nan)
        )

# 3. Numerical Variable Classification
# Integer variables: use nullable Int64 (supports missing values)
cols_entiers = [
    'sbp', 'dbp', 'hr', 'sat', 'rr',
    'gcs', 'pain', 'triage'
]

# Decimal variables: use nullable Float64 (supports missing values)
cols_decimaux = [
    'temp', 'cap_blood_sugar', 'hemocue', 'o2_flow',
    'pupil_right', 'pupil_left',
    'breathalyzer', 'duration_triage_ioa_min'
]

# 4️. Apply Numeric Formatting

# Integer conversion
for col in cols_entiers:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

# Float conversion
for col in cols_decimaux:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('Float64')

# 5️. Technical Column Cleanup (in-place to avoid recreating objects)
cols_to_remove = [
    'temp_date_x', 'temp_date_y',
    'annee_x', 'annee_y',
    'source_file', 'tri_raw'
]

df.drop(columns=[col for col in cols_to_remove if col in df.columns], inplace=True)

# 6. Dataset Dimension Monitoring
print(f"--- Harmonization completed: {len(df.columns)} columns retained ---")

#----------------------------------
# STRUCTURED DATA SUMMARY (No Free Text)
#----------------------------------

# Select only numerical and datetime columns for lightweight overview
cols_apercu = df.select_dtypes(exclude=['object']).columns

info_final = pd.DataFrame({
    'Type': df[cols_apercu].dtypes,
    'Completion Rate (%)': (df[cols_apercu].notna().mean() * 100).round(1)
})

print("\n📊 STRUCTURED DATA STATISTICS:")
display(info_final)  # Lightweight display (numeric/datetime only)

#----------------------------------
# 1. Dipstick / Hemocue Exploration (Lightweight Version)
#----------------------------------

cols_a_tester = ['urine_dipstick', 'hemocue', 'breathalyzer', 'o2_flow']
stats_list = []

for col in [c for c in cols_a_tester if c in df.columns]:
    nb_val = df[col].notna().sum()

    stats_list.append({
        'Column': col,
        'Non-missing values': nb_val,
        'Completion Rate (%)': round((nb_val / len(df)) * 100, 2),
        'Examples (first 3)': str(df[col].dropna().unique()[:3])
    })

# Display a single clean summary table instead of multiple print statements
display(pd.DataFrame(stats_list))

#----------------------------------
# 2. Blood Glucose Conversion (In-Place, No Duplication)
#----------------------------------

if 'cap_blood_sugar' in df.columns:
    # Rename column directly
    df.rename(columns={'cap_blood_sugar': 'cap_blood_sugar_hgt_g_L'}, inplace=True)

    # Convert from g/L to mmol/L using factor 5.55
    df['cap_blood_sugar_mmol_L'] = (
            pd.to_numeric(df['cap_blood_sugar_hgt_g_L'], errors='coerce') * 5.55
    ).round(2)

print("\n✅ Blood glucose conversion completed.")

# Safe verification (strictly limited output)
print("Verification of 5.55 conversion factor:")
display(df[['cap_blood_sugar_hgt_g_L', 'cap_blood_sugar_mmol_L']].dropna().head(5))

--- Harmonization completed: 32 columns retained ---

📊 STRUCTURED DATA STATISTICS:


,Type,Completion Rate (%)
date_adm_final,datetime64[ns],100.0
duration_triage_ioa_min,Float64,3.2
triage,Int64,100.0
year,int64,100.0
sbp,Int64,63.3
dbp,Int64,63.3
hr,Int64,63.1
temp,Float64,61.2
sat,Int64,62.7
rr,Int64,14.6


,Column,Non-missing values,Completion Rate (%),Examples (first 3)
0,urine_dipstick,11149,6.60,['Positive Sang' 'ECBU' 'Positive Nitrites']
1,hemocue,3580,2.12,"<FloatingArray>\n[11.0, 11.2, 11.8]\nLength: 3..."
2,breathalyzer,2830,1.67,"<FloatingArray>\n[2.19, 0.7, 1.17]\nLength: 3,..."
3,o2_flow,89170,52.77,"<FloatingArray>\n[2.0, 0.0, 6.0]\nLength: 3, d..."



✅ Blood glucose conversion completed.
Verification of 5.55 conversion factor:


,cap_blood_sugar_hgt_g_L,cap_blood_sugar_mmol_L
2,1.03,5.72
12,1.35,7.49
13,1.31,7.27
14,1.17,6.49
15,0.93,5.16


In [12]:
#---------------------------------------------------------
# DESCRIPTIVE STATISTICS SUMMARY
#---------------------------------------------------------

# 1️⃣ Strict exclusion list
free_text_cols = ['anam_ioa', 'atcd_ioa', 'chief_complaint', 'nda']
exclude_list = ['adm_date_time'] + free_text_cols

# Identify variables
numeric_cols = [
    c for c in df.select_dtypes(include=['number']).columns
    if c not in exclude_list
]

# Keep only "short" categorical variables (e.g., Hospital, Triage, etc.)
categorical_cols = [
    c for c in df.select_dtypes(include=['object', 'string', 'category']).columns
    if c not in exclude_list
]


# --- SECTION A: QUANTITATIVE VARIABLES ---
print("=" * 60)
print("📊 STATISTICAL SUMMARY: QUANTITATIVE VARIABLES")
print("=" * 60)

if numeric_cols:
    stats = df[numeric_cols].describe(percentiles=[.25, .5, .75]).T
    stats = stats.rename(columns={'25%': 'q25', '50%': 'median', '75%': 'q75'})
    stats['missing_%'] = (
        (df[numeric_cols].isna().sum() / len(df)) * 100
    ).round(1)

    # Use display() for a clean table format
    # Round values for better readability
    display(
        stats[['missing_%', 'count', 'min', 'median', 'mean', 'max']]
        .round(2)
    )
else:
    print("No numerical columns found.")


# --- SECTION B: CATEGORICAL VARIABLES ---
print("\n" + "=" * 60)
print("📋 STATISTICAL SUMMARY: CATEGORICAL VARIABLES")
print("=" * 60)

for col in categorical_cols:
    # SAFETY: Only print non–free-text variables
    # Limit output to the 20 most frequent categories
    counts = df[col].value_counts(dropna=False).head(20)
    percent = (
        df[col]
        .value_counts(dropna=False, normalize=True)
        .head(20) * 100
    ).round(1)

    cat_table = pd.concat([counts, percent], axis=1)
    cat_table.columns = ['N', '%']

    print(f"\n🔹 VARIABLE: {col}")
    display(cat_table)


# --- SECTION C: FREE TEXT AUDIT (No content displayed) ---
print("\n" + "=" * 60)
print("📝 FREE TEXT COLUMNS AUDIT (Content not displayed)")
print("=" * 60)

for col in free_text_cols:
    if col in df.columns:
        missing = df[col].isna().sum()
        print(
            f"{col:<15} | Missing values: {missing:,} "
            f"({(missing / len(df) * 100):.1f}%)"
        )

📊 STATISTICAL SUMMARY: QUANTITATIVE VARIABLES


,missing_%,count,min,median,mean,max
duration_triage_ioa_min,96.8,5348.0,2.0,5.0,6.224196,29.0
triage,0.0,168942.0,1.0,3.0,3.275225,5.0
year,0.0,168968.0,2022.0,2024.0,2023.532284,2025.0
sbp,36.7,107038.0,0.0,130.0,131.796026,1889.0
dbp,36.7,107031.0,0.0,77.0,77.964235,8263.0
hr,36.9,106680.0,0.0,78.0,80.285471,993.0
temp,38.8,103392.0,0.0,36.8,36.783026,41.2
sat,37.3,105944.0,0.0,98.0,97.543636,100.0
rr,85.4,24641.0,0.0,18.0,20.018587,388.0
o2_flow,47.2,89170.0,0.0,0.0,0.501161,99.0



📋 STATISTICAL SUMMARY: CATEGORICAL VARIABLES

🔹 VARIABLE: hospital


,N,%
hospital,,
PEL,168968,100.0



🔹 VARIABLE: transport


...


🔹 VARIABLE: date_adm_vitals


,N,%
date_adm_vitals,,
NaN,57884,34.3
2023-01-03 18:19:00,3,0.0
2023-02-11 18:30:00,2,0.0
2025-06-21 20:16:00,2,0.0
2023-03-22 10:51:00,2,0.0
2025-06-21 19:55:00,2,0.0
2023-06-22 18:26:00,2,0.0
2025-04-20 19:28:00,2,0.0
2022-02-02 19:06:00,2,0.0



🔹 VARIABLE: urine_dipstick


,N,%
urine_dipstick,,
NaN,157819,93.4
Positive Sang,2576,1.5
Négative,2494,1.5
ECBU,2175,1.3
Positive Leucocytes,1297,0.8
Beta HCG,1056,0.6
Positive Cétone,528,0.3
Protéines urinaires,512,0.3
Positive Nitrites,511,0.3



📝 FREE TEXT COLUMNS AUDIT (Content not displayed)
anam_ioa        | Missing values: 548 (0.3%)
atcd_ioa        | Missing values: 24,356 (14.4%)
chief_complaint | Missing values: 0 (0.0%)
nda             | Missing values: 0 (0.0%)


In [13]:
# ======================================================
# DATA QUALITY & VARIABLE INVESTIGATION
# ======================================================


# ======================================================
# 1️⃣ Blood Pressure Consistency Check (SBP / DBP)
# ======================================================

# Identify incomplete and complete blood pressure pairs
sbp_only = df[df['sbp'].notna() & df['dbp'].isna()]
dbp_only = df[df['dbp'].notna() & df['sbp'].isna()]
both_bp = df[df['sbp'].notna() & df['dbp'].notna()]

print(f"📊 BLOOD PRESSURE ANALYSIS (n = {len(df)})")
print("-" * 45)
print(f"✅ Complete pairs (SBP + DBP): {len(both_bp)} "
      f"({round(len(both_bp)/len(df)*100, 1)}%)")
print(f"⚠️ SBP only (DBP missing): {len(sbp_only)}")
print(f"⚠️ DBP only (SBP missing): {len(dbp_only)}")

# Inspect patients with SBP recorded but DBP missing
cols_to_display = ['sbp', 'dbp', 'temp', 'hr', 'sat']
display(sbp_only[cols_to_display])

# Interpretation:
# These blood pressures were clearly measured.
# The issue likely reflects a data entry problem rather than a true absence of measurement.
# Therefore, they should not automatically be considered as "not measured".


# ======================================================
# 2️⃣ Urine Dipstick Harmonization
# ======================================================

if 'urine_dipstick' in df.columns:

    # Basic standardization (lowercase + strip spaces)
    df['urine_dipstick'] = df['urine_dipstick'].str.lower().str.strip()

    # Create a cleaned version to preserve the original column
    df['urine_dipstick_clean'] = df['urine_dipstick'].copy()

    # Regex-based grouping (flexible pattern matching)
    replacements = {
        r'.*négatif.*': 'negative',
        r'.*négative.*': 'negative',
        r'.*positif$': 'positive_general',
        r'.*positive$': 'positive_general',
        r'.*sang.*': 'blood_positive',
        r'.*leuco.*': 'leukocytes_positive',
        r'.*nitrit.*': 'nitrites_positive',
        r'.*cétone.*': 'ketones_positive',
        r'.*ecbu.*': 'ecbu_ordered',
        r'.*protéine.*': 'protein_positive'
    }

    for pattern, replacement in replacements.items():
        df['urine_dipstick_clean'] = (
            df['urine_dipstick_clean']
            .str.replace(pattern, replacement, regex=True)
        )

    print("--- 🧪 Harmonized Urine Dipstick Levels ---")
    print(df['urine_dipstick_clean'].value_counts())


# ======================================================
# 3️⃣ Transport Mode Distribution
# ======================================================

transport_counts = df['transport'].value_counts(dropna=False)
transport_percent = (
    df['transport']
    .value_counts(normalize=True, dropna=False) * 100
).round(1)

transport_summary = pd.DataFrame({
    'Count': transport_counts,
    'Percentage (%)': transport_percent
})

print("📊 DISTRIBUTION OF TRANSPORT MODES")
display(transport_summary)


# ======================================================
# 4️⃣ Temporal Analysis of Transport Mode
# ======================================================

# Create monthly period variable
df['month_admission'] = df['date_adm_final'].dt.to_period('M').astype(str)

# Monthly cross-tabulation (raw counts)
transport_evolution = pd.crosstab(
    df['month_admission'],
    df['transport'],
    dropna=False
)

# Add monthly total column
transport_evolution['MONTH_TOTAL'] = transport_evolution.sum(axis=1)

print("📊 MONTHLY EVOLUTION OF TRANSPORT MODES (Raw Counts)")
display(transport_evolution)

# Monthly percentage distribution (relative proportions)
transport_pct_evol = (
    pd.crosstab(
        df['month_admission'],
        df['transport'],
        normalize='index'
    )
    .mul(100)
    .round(1)
)

print("\n📈 MONTHLY RELATIVE DISTRIBUTION OF TRANSPORT MODES (%)")
display(transport_pct_evol)

📊 BLOOD PRESSURE ANALYSIS (n = 168968)
---------------------------------------------
✅ Complete pairs (SBP + DBP): 107030 (63.3%)
⚠️ SBP only (DBP missing): 8
⚠️ DBP only (SBP missing): 1


,sbp,dbp,temp,hr,sat
462,107,<NA>,36.0,73,99
6696,134,<NA>,37.4,73,97
16279,126,<NA>,37.3,54,99
58506,0,<NA>,37.2,98,96
81792,0,<NA>,38.4,112,90
110861,133,<NA>,36.4,73,98
127199,0,<NA>,<NA>,80,100
134958,130,<NA>,36.6,97,100


--- 🧪 Harmonized Urine Dipstick Levels ---
urine_dipstick_clean
blood_positive         2576
negative               2494
ecbu_ordered           2175
leukocytes_positive    1297
beta hcg               1056
ketones_positive        528
protein_positive        512
nitrites_positive       511
Name: count, dtype: int64
📊 DISTRIBUTION OF TRANSPORT MODES


,Count,Percentage (%)
transport,,
Moyens personnels,97285,57.6
Pompiers,35258,20.9
Ambulance privée,20021,11.8
NaN,5631,3.3
Spontanée,5610,3.3
Ambulance publique,1104,0.7
Autres,1081,0.6
"SAMU, SMUR",1027,0.6
Ambulances privées,909,0.5


📊 MONTHLY EVOLUTION OF TRANSPORT MODES (Raw Counts)


transport,Ambulance privée,Ambulance publique,Ambulances privées,Appel SAMU Centre 15,Autres,Hélicoptère,Moyens personnels,Médecin traitant,Police,Pompiers,"SAMU, SMUR",SMUR,SMUR/Pompier,SOS médecin,Spontanée,Taxi,Vélo,NaN,MONTH_TOTAL
month_admission,,,,,,,,,,,,,,,,,,,
2022-01,0,0,361,1,0,0,0,80,52,846,0,7,0,38,2132,0,0,412,3929
2022-02,0,0,293,10,0,0,0,104,46,690,0,15,0,45,1928,0,0,442,3573
2022-03,136,9,247,8,7,3,929,85,37,765,6,15,0,27,1544,1,0,447,4266
2022-04,395,8,0,0,27,5,2727,0,0,818,18,0,0,0,0,6,0,27,4031
2022-05,371,20,1,0,34,11,2320,0,0,722,17,0,0,0,0,5,0,39,3540
2022-06,369,10,1,0,26,5,1909,0,0,837,34,0,0,0,1,1,0,65,3258
2022-07,426,10,4,1,33,12,2080,0,0,702,30,0,0,0,1,2,0,61,3362
2022-08,478,8,0,0,33,10,2124,0,0,709,25,0,0,0,1,0,0,54,3442
2022-09,466,10,0,0,22,7,2046,0,0,758,22,0,0,0,0,3,0,88,3422



📈 MONTHLY RELATIVE DISTRIBUTION OF TRANSPORT MODES (%)


transport,Ambulance privée,Ambulance publique,Ambulances privées,Appel SAMU Centre 15,Autres,Hélicoptère,Moyens personnels,Médecin traitant,Police,Pompiers,"SAMU, SMUR",SMUR,SMUR/Pompier,SOS médecin,Spontanée,Taxi,Vélo
month_admission,,,,,,,,,,,,,,,,,
2022-01,0.0,0.0,10.3,0.0,0.0,0.0,0.0,2.3,1.5,24.1,0.0,0.2,0.0,1.1,60.6,0.0,0.0
2022-02,0.0,0.0,9.4,0.3,0.0,0.0,0.0,3.3,1.5,22.0,0.0,0.5,0.0,1.4,61.6,0.0,0.0
2022-03,3.6,0.2,6.5,0.2,0.2,0.1,24.3,2.2,1.0,20.0,0.2,0.4,0.0,0.7,40.4,0.0,0.0
2022-04,9.9,0.2,0.0,0.0,0.7,0.1,68.1,0.0,0.0,20.4,0.4,0.0,0.0,0.0,0.0,0.1,0.0
2022-05,10.6,0.6,0.0,0.0,1.0,0.3,66.3,0.0,0.0,20.6,0.5,0.0,0.0,0.0,0.0,0.1,0.0
2022-06,11.6,0.3,0.0,0.0,0.8,0.2,59.8,0.0,0.0,26.2,1.1,0.0,0.0,0.0,0.0,0.0,0.0
2022-07,12.9,0.3,0.1,0.0,1.0,0.4,63.0,0.0,0.0,21.3,0.9,0.0,0.0,0.0,0.0,0.1,0.0
2022-08,14.1,0.2,0.0,0.0,1.0,0.3,62.7,0.0,0.0,20.9,0.7,0.0,0.0,0.0,0.0,0.0,0.0
2022-09,14.0,0.3,0.0,0.0,0.7,0.2,61.4,0.0,0.0,22.7,0.7,0.0,0.0,0.0,0.0,0.1,0.0


In [14]:
import pandas as pd
import numpy as np
from IPython.display import display

# ============================================================
# 1. HARMONIZED TRANSPORT MAPPING
# ============================================================

# Note: The values on the right MUST match the target categories in 'transport_order'
transport_mapping = {
    # PERSONAL TRANSPORT
    'Moyens personnels': 'Personal',
    'Spontanée': 'Personal',
    'Taxi': 'Personal',
    'Police': 'Personal',  # Often grouped with autonomous arrivals

    # EMERGENCY SERVICES
    'SAMU, SMUR': 'Emergency services',
    'SMUR': 'Emergency services',
    'Pompiers': 'Emergency services',
    'Hélicoptère': 'Emergency services',

    # AMBULANCE
    'Ambulance privée': 'Ambulance',
    'Ambulances privées': 'Ambulance',
    'Ambulance publique': 'Ambulance',

    # POST-MEDICAL ADVICE
    'SOS médecin': 'Post medical advice',
    'Médecin traitant': 'Post medical advice',
    'Appel SAMU Centre 15': 'Post medical advice',

    # OTHER / UNKNOWN
    'Autres': 'Unknown'
}

# ============================================================
# 2. CLEANING AND STANDARDIZATION
# ============================================================

raw_col = 'transport'
new_col = 'transport_grouped'

# Apply mapping
df[new_col] = df[raw_col].replace(transport_mapping)

# Replace missing or unmapped values with "Unknown"
df[new_col] = df[new_col].fillna('Unknown')

# Safety check: force any unexpected value into "Unknown"
expected_values = [
    'Personal',
    'Ambulance',
    'Post medical advice',
    'Emergency services',
    'Unknown'
]

df.loc[~df[new_col].isin(expected_values), new_col] = 'Unknown'

# ============================================================
# 3. CATEGORY ORDERING
# ============================================================

transport_order = [
    'Unknown',
    'Personal',
    'Ambulance',
    'Post medical advice',
    'Emergency services'
]

df[new_col] = pd.Categorical(
    df[new_col],
    categories=transport_order,
    ordered=True
)

# ============================================================
# 4. SUMMARY TABLE
# ============================================================

print(f"✅ Created variable: {new_col} (from: {raw_col})")
print("-" * 50)

summary = pd.DataFrame({
    'Count': df[new_col].value_counts().sort_index(),
    'Frequency (%)': (df[new_col].value_counts(normalize=True).sort_index() * 100).round(1)
})

display(summary)


✅ Created variable: transport_grouped (from: transport)
--------------------------------------------------


,Count,Frequency (%)
transport_grouped,,
Unknown,6719,4.0
Personal,103139,61.0
Ambulance,22034,13.0
Post medical advice,400,0.2
Emergency services,36676,21.7


In [15]:
# ==============================================================================
# CLINICAL PIPELINE
# ==============================================================================


# ==============================================================================
# 1. MEASUREMENT FLAGS — on raw data (outliers still present)
# ==============================================================================

def apply_measurement_flags(df):
    """
    Step 1 — Create is_*_measured flags on RAW data (before outlier removal).
    A value is considered measured even if aberrant.

    Special cases :
    - Blood pressure : measured if sbp OR dbp is present
    - O2             : measured if o2_flow is not NaN (0 = measured but off)
    - Pupils         : measured if at least one pupil value is present
    """

    # ── Blood pressure ─────────────────────────────────────────────────────────
    if "sbp" in df.columns and "dbp" in df.columns:
        df["is_bp_measured"] = (df["sbp"].notna() | df["dbp"].notna()).astype(int)

    # ── O2 ────────────────────────────────────────────────────────────────────
    if "o2_flow" in df.columns:
        df["is_o2_measured"] = df["o2_flow"].notna().astype(int)

    # ── Pupils ────────────────────────────────────────────────────────────────
    if "pupil_right" in df.columns and "pupil_left" in df.columns:
        df["is_pupils_measured"] = (
            df["pupil_right"].notna() | df["pupil_left"].notna()
        ).astype(int)

    # ── Standard vitals ────────────────────────────────────────────────────────
    standard_vitals = [
        "temp", "hr", "sat", "rr",
        "urine_dipstick_clean", "hemocue",
        "gcs", "cap_blood_sugar_mmol_L",
        "pain", "breathalyzer",
    ]
    for col in standard_vitals:
        if col in df.columns:
            df[f"is_{col}_measured"] = df[col].notna().astype(int)

    # ── Remove duplicates ──────────────────────────────────────────────────────
    df.drop(
        columns=["is_sbp_measured", "is_dbp_measured", "is_mbp_measured"],
        errors="ignore", inplace=True,
    )

    return df


# ==============================================================================
# 2. OUTLIER REMOVAL — after flags, before status
# ==============================================================================

def clean_vital_outliers(df):
    """
    Step 2 — Remove rows with physiologically impossible values.
    Flags (is_*_measured) are already set.
    No 'invalid' category needed since rows are dropped.
    """
    bounds = {
        "temp":                   (25,    43  ),
        "hr":                     (20,   300  ),
        "sat":                    (40,   100  ),
        "sbp":                    (40,   300  ),
        "dbp":                    (20,   200  ),
        "rr":                     ( 4,    60  ),
        "gcs":                    ( 3,    15  ),
        "pain":                   ( 0,    10  ),
        "cap_blood_sugar_mmol_L": ( 1.1,  33.3),
        "hemocue":                ( 5,    20  ),
        "pupil_right":            ( 1,     9  ),
        "pupil_left":             ( 1,     9  ),
        "o2_flow":                ( 0,    15  ),
    }

    n_before = len(df)

    for col, (lo, hi) in bounds.items():
        if col in df.columns:
            mask_outlier = df[col].notna() & ~df[col].between(lo, hi)
            n_outlier    = mask_outlier.sum()
            if n_outlier > 0:
                df = df[~mask_outlier].copy()
                print(f"  {col} : {n_outlier} rows removed "
                      f"(bounds [{lo}, {hi}]) — {len(df)} rows remaining")

    n_removed = n_before - len(df)
    print(f"\n  Total removed : {n_removed} rows "
          f"({100*n_removed/n_before:.1f}%)")

    return df


# ==============================================================================
# 3. CLINICAL STATUS — 2 cases per variable
# ==============================================================================

def _clean_numeric(series):
    """Convert a column to float, handling commas and stray characters."""
    return (
        series.astype(str)
              .str.replace(",", ".", regex=False)
              .str.replace(r"[^0-9.\-]+", "", regex=True)
              .replace("", np.nan)
              .astype(float)
    )


def _cut_with_flag(series, flag_col, bins, labels, df):
    """
    Apply pd.cut then handle 2 cases :
    - is_measured = 0 → 'not_measured'
    - is_measured = 1 → clinical label
    No 'invalid' case since outlier rows are dropped.
    """
    result = pd.cut(series, bins=bins, labels=labels).astype("object")
    if flag_col in df.columns:
        result[df[flag_col] == 0] = "not_measured"
    return result


def _select_with_flag(conditions, choices, default, flag_col, df):
    """
    Apply np.select then handle 2 cases :
    - is_measured = 0 → 'not_measured'
    - is_measured = 1 → clinical label
    No 'invalid' case since outlier rows are dropped.
    """
    result = np.select(conditions, choices, default=default)
    if flag_col in df.columns:
        result = np.where(
            df[flag_col] == 0,
            "not_measured", result,
        )
    return result


def apply_clinical_logic(df):
    """
    Step 3 — Compute status variables.

    2 possible values per status :
    - 'not_measured' : is_*_measured = 0 (never measured)
    - clinical label : is_*_measured = 1 and value valid
                       (outlier rows already removed in step 2)
    """

    # ── Clean numeric columns ──────────────────────────────────────────────────
    numeric_cols = [
        "temp", "hr", "sbp", "dbp", "sat", "rr",
        "o2_flow", "cap_blood_sugar_mmol_L",
        "hemocue", "gcs", "pain", "breathalyzer",
        "pupil_right", "pupil_left",
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = _clean_numeric(df[col])

    # ── Temperature ───────────────────────────────────────────────────────────
    if "temp" in df.columns:
        df["temp_status"] = _cut_with_flag(
            df["temp"], "is_temp_measured",
            bins=[0, 35.9, 38, 50],
            labels=["hypothermia", "normothermia", "hyperthermia"],
            df=df,
        )

    # ── Heart rate ────────────────────────────────────────────────────────────
    if "hr" in df.columns:
        df["hr_status"] = _cut_with_flag(
            df["hr"], "is_hr_measured",
            bins=[0, 49, 100, 300],
            labels=["bradycardia", "normocardia", "tachycardia"],
            df=df,
        )

    # ── Oxygen saturation ─────────────────────────────────────────────────────
    if "sat" in df.columns:
        df["sat_status"] = _cut_with_flag(
            df["sat"], "is_sat_measured",
            bins=[0, 87, 93, 100],
            labels=["severe hypoxia","hypoxia", "normal"],
            df=df,
        )

    # ── GCS ───────────────────────────────────────────────────────────────────
    if "gcs" in df.columns:
        df["gcs_status"] = _cut_with_flag(
            df["gcs"], "is_gcs_measured",
            bins=[0, 8, 12, 15],
            labels=["severe_impairment", "moderate_impairment", "normal"],
            df=df,
        )

    # ── Capillary blood sugar ─────────────────────────────────────────────────
    if "cap_blood_sugar_mmol_L" in df.columns:
        df["cap_blood_sugar_status"] = _cut_with_flag(
            df["cap_blood_sugar_mmol_L"], "is_cap_blood_sugar_mmol_L_measured",
            bins=[0, 3.8, 7.8, 100],
            labels=["hypoglycemia", "normoglycemia", "hyperglycemia"],
            df=df,
        )

    # ── Respiratory rate ──────────────────────────────────────────────────────
    if "rr" in df.columns:
        df["rr_status"] = _cut_with_flag(
            df["rr"], "is_rr_measured",
            bins=[0, 11, 20, 100],
            labels=["bradypnea", "normal", "tachypnea"],
            df=df,
        )

    # ── Pain ──────────────────────────────────────────────────────────────────
    if "pain" in df.columns:
        df["pain_status"] = _cut_with_flag(
            df["pain"], "is_pain_measured",
            bins=[-1, 0, 3, 6, 10],
            labels=["no_pain", "mild_pain", "moderate_pain", "severe_pain"],
            df=df,
        )

    # ── Hemocue ───────────────────────────────────────────────────────────────
    if "hemocue" in df.columns:
        df["hemocue_status"] = _cut_with_flag(
            df["hemocue"], "is_hemocue_measured",
            bins=[0, 7, 13, np.inf],
            labels=["severe_anemia", "moderate_anemia", "normal"],
            df=df,
        )

    # ── Urine dipstick ────────────────────────────────────────────────────────
    if "urine_dipstick_clean" in df.columns:
        result = np.where(
            df["urine_dipstick_clean"] == "negative", "negative", "positive"
        )
        if "is_urine_dipstick_clean_measured" in df.columns:
            result = np.where(
                df["is_urine_dipstick_clean_measured"] == 0,
                "not_measured", result,
            )
        df["urine_dipstick_clean_status"] = result

    # ── Breathalyzer ──────────────────────────────────────────────────────────
    if "breathalyzer" in df.columns:
        result = np.select(
            [df["breathalyzer"] > 0, df["breathalyzer"] == 0],
            ["positive", "negative"],
            default="not_measured",
        )
        if "is_breathalyzer_measured" in df.columns:
            result = np.where(
                df["is_breathalyzer_measured"] == 0,
                "not_measured", result,
            )
        df["breathalyzer_status"] = result

    # ── Blood pressure ────────────────────────────────────────────────────────
    if "sbp" in df.columns and "dbp" in df.columns:
        df["mbp"]  = ((df["sbp"] + 2 * df["dbp"]) / 3).round(1)
        cond_hypo  = (df["sbp"] < 90) | (df["dbp"] < 60) | (df["mbp"] < 65)
        cond_hyper = (df["sbp"] >= 140) | (df["dbp"] >= 90)

        df["bp_status"] = _select_with_flag(
            conditions = [cond_hypo, cond_hyper],
            choices    = ["hypotension", "hypertension"],
            default    = "normotension",
            flag_col   = "is_bp_measured",
            df         = df,
        )

    # ── O2 flow ───────────────────────────────────────────────────────────────
    if "o2_flow" in df.columns:
        df["o2_flow_status"] = _select_with_flag(
            conditions = [df["o2_flow"] > 0, df["o2_flow"] == 0],
            choices    = ["on", "off"],
            default    = "not_measured",
            flag_col   = "is_o2_measured",
            df         = df,
        )

    # ── Pupils ────────────────────────────────────────────────────────────────
    if "pupil_right" in df.columns and "pupil_left" in df.columns:
        pr = df["pupil_right"]
        pl = df["pupil_left"]

        # If only one value → assume both equal
        pr_filled = pr.fillna(pl)
        pl_filled = pl.fillna(pr)

        # Anisocoria
        diff = (pr_filled - pl_filled).abs()
        df["anisocoria_status"] = np.where(diff >= 0.5, "yes", "no")

        # Pupils status — based on largest pupil
        max_pupil = np.maximum(pr_filled, pl_filled)
        df["pupils_status"] = np.select(
            [
                max_pupil < 2,
                max_pupil.between(2, 4),
                max_pupil > 4,
            ],
            ["myosis", "normal", "mydriasis"],
            default="normal",
        )

        # Not measured only — no invalid since rows are dropped
        mask_not_measured = (
            df["is_pupils_measured"] == 0
            if "is_pupils_measured" in df.columns
            else pd.Series(False, index=df.index)
        )
        df.loc[mask_not_measured,
               ["anisocoria_status", "pupils_status"]] = "not_measured"

    return df


# ==============================================================================
# 4. DASHBOARD
# ==============================================================================

def get_complete_dashboard(df):
    """Descriptive summary — numeric + categorical."""
    exclude     = ["visit_id", "adm_date_time", "date_hour_adm",
                   "nda", "anam_ioa", "chief_complaint"]
    target_cols = [c for c in df.columns if c not in exclude]

    # Numeric
    num_df      = df[target_cols].select_dtypes(include=[np.number])
    num_summary = num_df.agg(["mean", "std", "min", "max"]).T
    num_summary["Missing %"] = (num_df.isna().sum() / len(df) * 100).round(2)

    # Categorical (<20 modalities)
    cat_df   = df[target_cols].select_dtypes(exclude=[np.number])
    cat_cols = [c for c in cat_df.columns if df[c].nunique() < 20]
    rows     = []
    for col in cat_cols:
        counts = df[col].value_counts(dropna=False)
        perc   = df[col].value_counts(dropna=False, normalize=True) * 100
        for level in counts.index:
            rows.append({
                "Variable": col,
                "Category": level,
                "N":        counts[level],
                "%":        round(perc[level], 1),
            })

    return num_summary.round(2), pd.DataFrame(rows)


# ==============================================================================
# 5. FULL PIPELINE
# ==============================================================================

def run_full_clinical_pipeline(df):
    """
    Full pipeline in the correct order :

    1. Measurement flags  → on raw data (outliers still present)
    2. Outlier removal    → rows with aberrant values removed
    3. Clinical status    → 2 cases : not_measured / clinical label
    4. Dashboard
    """
    print("\n===== STEP 1 — MEASUREMENT FLAGS (raw data) =====")
    df = apply_measurement_flags(df)

    print("\n===== STEP 2 — OUTLIER REMOVAL =====")
    df = clean_vital_outliers(df)

    print("\n===== STEP 3 — CLINICAL STATUS =====")
    df = apply_clinical_logic(df)

    print("\n===== STEP 4 — DASHBOARD =====")
    num_summary, cat_summary = get_complete_dashboard(df)
    print("\n📊 Numeric summary:")
    display(num_summary)
    print("\n📋 Categorical summary:")
    display(cat_summary)

    print("\n===== PIPELINE COMPLETE =====")
    return df


# ==============================================================================
# 6. CALL
# ==============================================================================

df = run_full_clinical_pipeline(df)


===== STEP 1 — MEASUREMENT FLAGS (raw data) =====



===== STEP 2 — OUTLIER REMOVAL =====


  temp : 49 rows removed (bounds [25, 43]) — 168919 rows remaining


  hr : 108 rows removed (bounds [20, 300]) — 168811 rows remaining
  sat : 139 rows removed (bounds [40, 100]) — 168672 rows remaining


  sbp : 145 rows removed (bounds [40, 300]) — 168527 rows remaining


...


📊 Numeric summary:


,mean,std,min,max,Missing %
duration_triage_ioa_min,6.23,3.75,2.00,29.0,96.83
triage,3.28,0.92,1.00,5.0,0.02
year,2023.53,1.14,2022.00,2025.0,0.00
sbp,131.91,22.62,45.00,266.0,37.02
dbp,77.95,14.88,20.00,168.0,37.02
hr,80.16,16.41,20.00,225.0,37.22
temp,36.80,0.56,27.00,40.7,39.16
sat,97.67,2.11,45.00,100.0,37.63
rr,18.50,6.17,4.00,60.0,85.85
o2_flow,0.24,1.34,0.00,15.0,47.58



📋 Categorical summary:


,Variable,Category,N,%
0,hospital,PEL,167290,100.0
1,transport,Moyens personnels,96629,57.8
2,transport,Pompiers,34721,20.8
3,transport,Ambulance privée,19713,11.8
4,transport,NaN,5573,3.3
...,...,...,...,...
102,anisocoria_status,yes,2666,1.6
103,pupils_status,not_measured,145332,86.9
104,pupils_status,normal,21394,12.8
105,pupils_status,myosis,302,0.2



===== PIPELINE COMPLETE =====


In [16]:
# Vérifier si les colonnes pupilles existent dans ton df
print("pupil_right" in df.columns)
print("pupil_left" in df.columns)

# Vérifier si les colonnes ont des valeurs
print(df["pupil_right"].notna().sum())
print(df["pupil_left"].notna().sum())

# Vérifier si le flag a été créé
print("is_pupils_measured" in df.columns)

# Vérifier si apply_measurement_flags a bien tourné
print([c for c in df.columns if "pupil" in c])

True
True
21888
21860
True
['pupil_right', 'pupil_left', 'is_pupils_measured', 'pupils_status']


In [17]:

# =========================================================
# 5. COLUMN INVENTORY + ORDERING
# =========================================================
inventory = pd.DataFrame({
    'Dtype': df.dtypes,
    'Non-Null Count': df.count(),
    'Fill Rate %': (df.count() / len(df) * 100).round(1),
    'Example Value': df.iloc[0]
}).sort_index()

print(f"✅ Your DataFrame currently contains {len(df.columns)} columns.")
display(inventory)

print("\n📝 COLUMN LIST:")
print(df.columns.tolist())


✅ Your DataFrame currently contains 64 columns.


,Dtype,Non-Null Count,Fill Rate %,Example Value
admission_summary_ioa,object,35637,21.3,NaN
anam_ioa,object,166754,99.7,"Suite aune rixe, plaie doigt et temporale et o..."
anisocoria_status,object,167290,100.0,not_measured
atcd_ioa,object,143127,85.6,"Depression sous Deroxat, xanax\r\nOH"
bp_status,object,167290,100.0,not_measured
...,...,...,...,...
ttt_adm_ioa_file,object,44011,26.3,NaN
urine_dipstick,object,10958,6.6,NaN
urine_dipstick_clean,object,10958,6.6,NaN
urine_dipstick_clean_status,object,167290,100.0,not_measured



📝 COLUMN LIST:
['nda', 'date_adm_final', 'hospital', 'transport', 'date_hour_triage_begin', 'date_hour_triage_end', 'duration_triage_ioa_min', 'chief_complaint', 'triage', 'triage_raw', 'atcd_ioa', 'anam_ioa', 'year', 'ttt_adm_ioa_file', 'admission_summary_ioa', 'evolution_ioa', 'date_adm_vitals', 'urine_dipstick', 'sbp', 'dbp', 'hr', 'temp', 'sat', 'rr', 'o2_flow', 'cap_blood_sugar_hgt_g_L', 'hemocue', 'gcs', 'pain', 'breathalyzer', 'pupil_right', 'pupil_left', 'cap_blood_sugar_mmol_L', 'urine_dipstick_clean', 'month_admission', 'transport_grouped', 'is_bp_measured', 'is_o2_measured', 'is_pupils_measured', 'is_temp_measured', 'is_hr_measured', 'is_sat_measured', 'is_rr_measured', 'is_urine_dipstick_clean_measured', 'is_hemocue_measured', 'is_gcs_measured', 'is_cap_blood_sugar_mmol_L_measured', 'is_pain_measured', 'is_breathalyzer_measured', 'temp_status', 'hr_status', 'sat_status', 'gcs_status', 'cap_blood_sugar_status', 'rr_status', 'pain_status', 'hemocue_status', 'urine_dipstick_c

In [18]:
# 1. Définition des groupes logiques
cols_admin = [
    'nda', 'date_adm_final', 'hospital', 'transport_grouped'
]

cols_ioa = [
    'date_hour_triage_begin', 'date_hour_triage_end', 'duration_triage_ioa_min', 'chief_complaint', 'triage', 'triage_raw', 'atcd_ioa', 'anam_ioa', 'evolution_ioa'
]

cols_vitals = [
    'date_hour_vitals',
    # BP
    'sbp', 'dbp', 'mbp', 'bp_status', 'is_bp_measured',
    # Heart rate
    'hr', 'hr_status', 'is_hr_measured',
    # Temperature
    'temp', 'temp_status', 'is_temp_measured',
    # Oxygène / Respiratory rate
    'sat', 'sat_status', 'is_sat_measured',
    'rr', 'rr_status', 'is_rr_measured',
    'o2_flow', 'o2_flow_status', 'is_o2_measured',
    # Neuro and blood sugar
    'gcs', 'gcs_status', 'is_gcs_measured',
    'cap_blood_sugar_mmol_L', 'cap_blood_sugar_status', 'is_cap_blood_sugar_mmol_L_measured',
    # Pupils
    'pupil_right', 'pupil_left', 'pupils_status', 'anisocoria_status', 'is_pupils_measured',
    # others
    'urine_dipstick_clean', 'is_urine_dipstick_clean_measured', 'urine_dipstick_clean_status',
    'pain', 'pain_status', 'is_pain_measured',
    'breathalyzer', 'breathalyzer_status', 'is_breathalyzer_measured',
    'hemocue', 'is_hemocue_measured', 'hemocue_status'
]

# 2. Application de l'ordre (en vérifiant que les colonnes existent)
ordered_columns = [c for c in (cols_admin + cols_ioa + cols_vitals) if c in df.columns]
df = df[ordered_columns]

print(f"✅ Reorgonized dataset : {len(df.columns)} colums sorted.")

✅ Reorgonized dataset : 56 colums sorted.


In [19]:
# --- Affichage détaillé pour validation ---
print("\n--- 📂 STRUCTURE DU DATASET ---")

# On boucle sur tes listes définies pour voir ce qui a été conservé
groups_to_check = {
    "ADMIN": cols_admin,
    "DOSSIER IOA": cols_ioa,
    "PARAMÈTRES VITAUX": cols_vitals
}

for label, col_list in groups_to_check.items():
    present = [c for c in col_list if c in df.columns]
    missing = [c for c in col_list if c not in df.columns]

    print(f"\n🔹 {label} ({len(present)} colonnes présentes)")
    for c in present:
        # On affiche un petit aperçu du type pour vérifier le nettoyage
        print(f"   - {c:<30} | {df[c].dtype}")

    if missing:
        print(f"   ⚠️ Manquantes dans le DF : {missing}")

print("\n--- 🚦 APERÇU DES PREMIÈRES LIGNES ---")
display(df.head())


--- 📂 STRUCTURE DU DATASET ---

🔹 ADMIN (4 colonnes présentes)
   - nda                            | object
   - date_adm_final                 | datetime64[ns]
   - hospital                       | object
   - transport_grouped              | category

🔹 DOSSIER IOA (9 colonnes présentes)
   - date_hour_triage_begin         | object
   - date_hour_triage_end           | object
   - duration_triage_ioa_min        | Float64
   - chief_complaint                | object
   - triage                         | Int64
   - triage_raw                     | object
   - atcd_ioa                       | object
   - anam_ioa                       | object
   - evolution_ioa                  | object

🔹 PARAMÈTRES VITAUX (43 colonnes présentes)
   - sbp                            | float64
   - dbp                            | float64
   - mbp                            | float64
   - bp_status                      | object
   - is_bp_measured                 | int64
   - hr                        

,nda,date_adm_final,hospital,transport_grouped,date_hour_triage_begin,date_hour_triage_end,duration_triage_ioa_min,chief_complaint,triage,triage_raw,atcd_ioa,anam_ioa,evolution_ioa,sbp,dbp,mbp,bp_status,is_bp_measured,hr,hr_status,is_hr_measured,temp,temp_status,is_temp_measured,sat,sat_status,is_sat_measured,rr,rr_status,is_rr_measured,o2_flow,o2_flow_status,is_o2_measured,gcs,gcs_status,is_gcs_measured,cap_blood_sugar_mmol_L,cap_blood_sugar_status,is_cap_blood_sugar_mmol_L_measured,pupil_right,pupil_left,pupils_status,anisocoria_status,is_pupils_measured,urine_dipstick_clean,is_urine_dipstick_clean_measured,urine_dipstick_clean_status,pain,pain_status,is_pain_measured,breathalyzer,breathalyzer_status,is_breathalyzer_measured,hemocue,is_hemocue_measured,hemocue_status
0,22030011617,2022-01-01 00:08:00,PEL,Emergency services,2022-01-01 00:19:00,2022-01-01 00:29:00,10.0,Victime d'agression physique,3,Urgent (médecin <1h),"Depression sous Deroxat, xanax\r\nOH","Suite aune rixe, plaie doigt et temporale et o...",NaN,NaN,NaN,NaN,not_measured,0,NaN,not_measured,0,NaN,not_measured,0,NaN,not_measured,0,NaN,not_measured,0,NaN,not_measured,0,NaN,not_measured,0,NaN,not_measured,0,NaN,NaN,not_measured,not_measured,0,NaN,0,not_measured,NaN,not_measured,0,NaN,not_measured,0,NaN,0,not_measured
1,22030011619,2022-01-01 00:17:00,PEL,Post medical advice,2022-01-01 00:28:00,2022-01-01 00:33:00,5.0,Détresse respiratoire aiguë majeure (FR >40/mi...,2,Très urgent (médecin <20min),"Surdité, cytolyse hépatique, glaucome, polyart...",Patiente adressé par sos médecin pour detresse...,"01/01/2022 06:15 - MAUXION Anna, Infirmièr(e)...",94.0,78.0,83.3,normotension,1,108.0,tachycardia,1,36.8,normothermia,1,94.0,normal,1,22.0,tachypnea,1,2.0,on,1,11.0,moderate_impairment,1,NaN,not_measured,0,NaN,NaN,not_measured,not_measured,0,NaN,0,not_measured,0.0,no_pain,1,NaN,not_measured,0,NaN,0,not_measured
2,22030011622,2022-01-01 00:22:00,PEL,Personal,2022-01-01 00:34:00,2022-01-01 00:40:00,6.0,Mouvements involontaires anormaux,4,Peu urgent (médecin <2h),ras,Patient emmené par sa maman pour crise d'angoi...,NaN,117.0,64.0,81.7,normotension,1,98.0,normocardia,1,36.8,normothermia,1,100.0,normal,1,16.0,normal,1,0.0,off,1,15.0,normal,1,5.72,normoglycemia,1,NaN,NaN,not_measured,not_measured,0,NaN,0,not_measured,0.0,no_pain,1,NaN,not_measured,0,NaN,0,not_measured
3,22030011625,2022-01-01 00:31:00,PEL,Personal,2022-01-01 00:49:00,2022-01-01 00:52:00,3.0,Demande de certificat médical,4,Peu urgent (médecin <2h),ras,"Allégation d'agression par son conjoint, consu...",NaN,137.0,99.0,111.7,hypertension,1,86.0,normocardia,1,36.4,normothermia,1,98.0,normal,1,NaN,not_measured,0,0.0,off,1,15.0,normal,1,NaN,not_measured,0,NaN,NaN,not_measured,not_measured,0,NaN,0,not_measured,0.0,no_pain,1,NaN,not_measured,0,NaN,0,not_measured
4,22030011629,2022-01-01 00:35:00,PEL,Personal,2022-01-01 00:42:00,2022-01-01 00:47:00,5.0,Anxiété sans troubles du comportement,4,Peu urgent (médecin <2h),hta traitée- IRA recente\r\ncontraception : st...,pec par les pompiers pour crise de panique se ...,NaN,NaN,NaN,NaN,not_measured,0,NaN,not_measured,0,NaN,not_measured,0,NaN,not_measured,0,NaN,not_measured,0,NaN,not_measured,0,NaN,not_measured,0,NaN,not_measured,0,NaN,NaN,not_measured,not_measured,0,NaN,0,not_measured,NaN,not_measured,0,NaN,not_measured,0,NaN,0,not_measured


In [20]:
df.to_csv('Datasets/df_ioafile_paramvit_clean.csv', index=False)